# Checkpoint C2: Retriever.py — Hybrid Retrieval
Kết quả chạy retriever.py với vector search + keyword fallback.

In [1]:
import sys
import shutil
from pathlib import Path

TEMPLATE_SKILL_DIR = Path("../../templates/skills/hr-policy-qa-skill").resolve()
OUTPUT_SKILL_DIR = Path("../../outputs/skills/hr-policy-qa-skill").resolve()

# Sao chép retriever.py sang outputs
src_retriever = TEMPLATE_SKILL_DIR / "scripts" / "retriever.py"
dst_retriever = OUTPUT_SKILL_DIR / "scripts" / "retriever.py"
if src_retriever.exists():
    shutil.copy(src_retriever, dst_retriever)
    print(f"✓ Đã sao chép retriever.py sang: {dst_retriever}")

scripts_path = (OUTPUT_SKILL_DIR / "scripts").resolve()
if str(scripts_path) not in sys.path:
    sys.path.insert(0, str(scripts_path))
print(f'Scripts path học viên: {scripts_path}')
print(f'Scripts dir exists: {scripts_path.exists()}')
if scripts_path.exists():
    print(f'Files: {list(scripts_path.glob("*.py"))}')


✓ Đã sao chép retriever.py sang: ../../outputs/skills/hr-policy-qa-skill/scripts/retriever.py
Scripts path học viên: /Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-02/session-04-knowledge-base-rag-hr-policy-qa/outputs/skills/hr-policy-qa-skill/scripts
Scripts dir exists: True
Files: [PosixPath('/Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-02/session-04-knowledge-base-rag-hr-policy-qa/outputs/skills/hr-policy-qa-skill/scripts/chunker.py'), PosixPath('/Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-02/session-04-knowledge-base-rag-hr-policy-qa/outputs/skills/hr-policy-qa-skill/scripts/retriever.py')]


In [2]:
# Kiểm tra các dependency cho vector search
VECTOR_SEARCH_AVAILABLE = False

try:
    from sentence_transformers import SentenceTransformer
    import chromadb
    VECTOR_SEARCH_AVAILABLE = True
    print('chromadb + sentence-transformers: SẴN SÀNG')
    print(f'  chromadb version: {chromadb.__version__}')
except ImportError as e:
    print(f'Vector search không khả dụng: {e}')
    print('Cần cài đặt: pip install chromadb sentence-transformers')
    print('==> Sử dụng keyword fallback thay thế.')

/Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


chromadb + sentence-transformers: SẴN SÀNG
  chromadb version: 1.5.9


In [3]:
from pathlib import Path
import re

# Load 4 file chính sách HR từ thư mục outputs học viên
policy_dir = Path('../../outputs/skills/hr-policy-qa-skill/kb/hr-policies').resolve()
policy_files = sorted(policy_dir.glob('*.md'))

policies = {}
for f in policy_files:
    content = f.read_text(encoding='utf-8')
    policies[f.name] = content
    print(f'Đã tải: {f.name} ({len(content)} ký tự)')

print(f'\nTổng cộng: Đã tải {len(policies)} file chính sách')

# Chunker đơn giản theo tiêu đề heading
chunks = []
for fname, content in policies.items():
    # Tách theo tiêu đề ## (level 2)
    sections = re.split(r'(^## .+$)', content, flags=re.MULTILINE)
    
    # Phần đầu tiên là header + giới thiệu
    header = sections[0].strip()
    doc_id_match = re.search(r'doc_id:\s*(\S+)', header)
    doc_id = doc_id_match.group(1) if doc_id_match else fname
    
    chunks.append({
        'source': fname,
        'doc_id': doc_id,
        'heading': 'Header',
        'text': header[:500],
        'full_text': header
    })
    
    # Các phần còn lại: cặp tiêu đề + nội dung
    i = 1
    while i < len(sections) - 1:
        heading = sections[i].strip()
        body = sections[i + 1].strip()
        heading_title = re.sub(r'^#+\s*', '', heading)
        chunks.append({
            'source': fname,
            'doc_id': doc_id,
            'heading': heading_title,
            'text': (heading + '\n' + body)[:500],
            'full_text': heading + '\n' + body
        })
        i += 2

print(f'\nTổng số chunks đã tạo: {len(chunks)}')
for c in chunks[:5]:
    print(f'  [{c["source"]}] {c["heading"]} ({len(c["text"])} ký tự)')
print('  ...')


Đã tải: policy-allowance.md (2418 ký tự)
Đã tải: policy-leave.md (2631 ký tự)
Đã tải: policy-seniority.md (2257 ký tự)
Đã tải: policy-training.md (3232 ký tự)

Tổng cộng: Đã tải 4 file chính sách

Tổng số chunks đã tạo: 20
  [policy-allowance.md] Header (287 ký tự)
  [policy-allowance.md] 1. Phụ cấp ăn trưa (500 ký tự)
  [policy-allowance.md] 2. Phụ cấp đi lại (500 ký tự)
  [policy-allowance.md] 3. Phụ cấp điện thoại (500 ký tự)
  [policy-leave.md] Header (291 ký tự)
  ...


In [4]:
def keyword_search(query, chunks, top_k=3):
    """
    Simple TF-IDF-like keyword search.
    Tokenize query and chunks into words, count matching terms,
    normalize by chunk length.
    """
    # Tách từ tiếng Việt đơn giản: viết thường, tách theo ký tự chữ và số
    query_terms = set(re.findall(r'\w+', query.lower()))
    
    results = []
    for chunk in chunks:
        chunk_terms = re.findall(r'\w+', chunk['text'].lower())
        chunk_term_set = set(chunk_terms)
        
        # Đếm các từ khớp
        matches = query_terms & chunk_term_set
        
        # Tính điểm số: cân bằng giữa precision và recall
        if len(query_terms) > 0 and len(chunk_term_set) > 0:
            recall = len(matches) / len(query_terms)
            precision = len(matches) / len(chunk_term_set)
            tf_boost = sum(chunk_terms.count(t) for t in matches) / max(len(chunk_terms), 1)
            score = recall * 0.5 + precision * 0.3 + tf_boost * 0.2
        else:
            score = 0.0
        
        results.append({
            'source': chunk['source'],
            'doc_id': chunk['doc_id'],
            'heading': chunk['heading'],
            'score': round(score, 4),
            'matched_terms': list(matches),
            'snippet': chunk['text'][:200]
        })
    
    # Sắp xếp theo điểm giảm dần
    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:top_k]

print('Hàm keyword_search() đã được định nghĩa.')
print('Input: query string, list of chunks, top_k')
print('Output: top-k kết quả kèm điểm số, các từ khớp, đoạn trích')

Hàm keyword_search() đã được định nghĩa.
Input: query string, list of chunks, top_k
Output: top-k kết quả kèm điểm số, các từ khớp, đoạn trích


In [5]:
# Test 1: In-scope direct query
query1 = 'Quy dinh nghi phep nam?'
print(f'Query: "{query1}"')
print('=' * 60)

results1 = keyword_search(query1, chunks, top_k=3)
for i, r in enumerate(results1, 1):
    print(f'\n#{i} [Score: {r["score"]}] Source: {r["source"]}')
    print(f'   Heading: {r["heading"]}')
    print(f'   Matched: {r["matched_terms"]}')
    print(f'   Snippet: {r["snippet"][:150]}...')

print('\n' + '-' * 60)

Query: "Quy dinh nghi phep nam?"

#1 [Score: 0.3405] Source: policy-leave.md
   Heading: Header
   Matched: ['nghi', 'phep', 'nam']
   Snippet: ---
doc_id: POL-LEAVE-001
title: Chinh sach nghi phep nam, nghi om va nghi thai san
version: v2.1
effective_date: 2026-01-01
owner: Phong Nhan su
acce...

#2 [Score: 0.1112] Source: policy-training.md
   Heading: 5. Đào tạo nội bộ
   Matched: ['quy']
   Snippet: ## 5. Đào tạo nội bộ
### 5.1 Buổi chia sẻ kiến thức

- Mỗi bộ phận tổ chức ít nhất 1 buổi chia sẻ kiến thức hàng quý.
- Diễn giả nội bộ được thưởng 50...

#3 [Score: 0.11] Source: policy-leave.md
   Heading: 4. Các loại nghỉ khác
   Matched: ['quy']
   Snippet: ## 4. Các loại nghỉ khác
| Loại nghỉ | Số ngày | Lương |
| --- | ---: | --- |
| Nghỉ việc riêng (kết hôn, tang) | 3 ngày | 100% |
| Nghỉ lễ, tết | The...

------------------------------------------------------------


In [6]:
# Test 2: In-scope query với điều kiện cụ thể
query2 = 'Phu cap dien thoai cho nhan vien thu viec'
print(f'Query: "{query2}"')
print('=' * 60)

results2 = keyword_search(query2, chunks, top_k=3)
for i, r in enumerate(results2, 1):
    print(f'\n#{i} [Score: {r["score"]}] Source: {r["source"]}')
    print(f'   Heading: {r["heading"]}')
    print(f'   Matched: {r["matched_terms"]}')
    print(f'   Snippet: {r["snippet"][:150]}...')

# Xác minh kết quả hàng đầu có thuộc chính sách phụ cấp không
if 'allowance' in results2[0]['source'] or 'ALLOW' in results2[0]['doc_id']:
    print('\nCHÍNH XÁC: Kết quả hàng đầu lấy từ chính sách phụ cấp.')
else:
    print('\nLƯU Ý: Kết quả hàng đầu có thể không liên quan nhất. Vector search sẽ cải thiện điều này.')

print('\n' + '-' * 60)

Query: "Phu cap dien thoai cho nhan vien thu viec"

#1 [Score: 0.3286] Source: policy-allowance.md
   Heading: Header
   Matched: ['phu', 'thoai', 'nhan', 'cap', 'dien']
   Snippet: ---
doc_id: POL-ALLOW-001
title: Chinh sach phu cap an trua, di lai va dien thoai
version: v1.3
effective_date: 2026-01-01
owner: Phong Nhan su
access...

#2 [Score: 0.0709] Source: policy-training.md
   Heading: Header
   Matched: ['nhan']
   Snippet: ---
doc_id: POL-TRAIN-001
title: Chinh sach dao tao va phat trien nhan luc
version: v1.1
effective_date: 2026-03-01
owner: Phong Nhan su
access_level:...

#3 [Score: 0.0679] Source: policy-seniority.md
   Heading: Header
   Matched: ['nhan']
   Snippet: ---
doc_id: POL-SENIOR-001
title: Chinh sach tham nien va thuong tham nien
version: v1.0
effective_date: 2026-01-01
owner: Phong Nhan su
access_level:...

CHÍNH XÁC: Kết quả hàng đầu lấy từ chính sách phụ cấp.

------------------------------------------------------------


In [7]:
# Test 3: Out-of-scope query
query3 = 'Bao hiem xa hoi'
print(f'Query: "{query3}" (ngoài phạm vi)')
print('=' * 60)

results3 = keyword_search(query3, chunks, top_k=3)
for i, r in enumerate(results3, 1):
    print(f'\n#{i} [Score: {r["score"]}] Source: {r["source"]}')
    print(f'   Heading: {r["heading"]}')
    print(f'   Matched: {r["matched_terms"]}')

# Kiểm tra: điểm số cho câu ngoài phạm vi phải thấp
max_score = results3[0]['score'] if results3 else 0
if max_score < 0.1:
    print(f'\nTỐT: Điểm tối đa là {max_score} (< 0.1) — xác định chính xác là ngoài phạm vi.')
elif max_score < 0.3:
    print(f'\nTẠM ĐƯỢC: Điểm tối đa là {max_score} — tương đối thấp, có khả năng ngoài phạm vi.')
else:
    print(f'\nLƯU Ý: Điểm tối đa là {max_score} — keyword search có thể tạo ra kết quả khớp giả.')
    print('Vector search sẽ đem lại hiểu biết ngữ nghĩa tốt hơn.')

print('\n' + '-' * 60)

Query: "Bao hiem xa hoi" (ngoài phạm vi)

#1 [Score: 0.1335] Source: policy-training.md
   Heading: 5. Đào tạo nội bộ
   Matched: ['bao']

#2 [Score: 0.132] Source: policy-allowance.md
   Heading: 1. Phụ cấp ăn trưa
   Matched: ['xa']

#3 [Score: 0.0] Source: policy-allowance.md
   Heading: Header
   Matched: []

TẠM ĐƯỢC: Điểm tối đa là 0.1335 — tương đối thấp, có khả năng ngoài phạm vi.

------------------------------------------------------------


### Chạy retriever.py qua dòng lệnh CLI

Kiểm tra xem script retriever của học viên có thể chạy thành công từ CLI và trả về kết quả không.

In [8]:
# Chạy retriever.py qua CLI cho câu hỏi trong phạm vi
!python ../../outputs/skills/hr-policy-qa-skill/scripts/retriever.py --query "Nhân viên chính thức được bao nhiêu ngày phép năm?" --top-k 3


config_sentence_transformers.json: 100%|███████| 122/122 [00:00<00:00, 329kB/s]
README.md: 100%|██████████████████████████| 3.89k/3.89k [00:00<00:00, 16.6MB/s]
sentence_bert_config.json: 100%|█████████████| 53.0/53.0 [00:00<00:00, 310kB/s]
config.json: 100%|████████████████████████████| 645/645 [00:00<00:00, 3.54MB/s]
model.safetensors: 100%|████████████████████| 471M/471M [00:08<00:00, 55.9MB/s]
tokenizer_config.json: 100%|██████████████████| 526/526 [00:00<00:00, 1.05MB/s]
tokenizer.json: 100%|█████████████████████| 9.08M/9.08M [00:01<00:00, 7.22MB/s]
config.json: 100%|████████████████████████████| 190/190 [00:00<00:00, 1.36MB/s]
[retriever] Chunks file not found: ./kb/chunks.json
Không tìm thấy thông tin liên quan cho: "Nhân viên chính thức được bao nhiêu ngày phép năm?"
Phương pháp: none | Kết quả: 0 | Gợi ý: từ chối trả lời (out-of-scope).

--- RAW JSON ---
{
  "results": [],
  "method": "none",
  "refused": true,
  "query": "Nhân viên chính thức được bao nhiêu ngày phép năm?"
}


In [9]:
print('=' * 60)
print('TỔNG KẾT CHECKPOINT C2')
print('=' * 60)
print()
print(f'Chính sách đã tải:    {len(policies)} file')
print(f'Chunks đã tạo:        {len(chunks)} chunks')
print(f'Tìm kiếm vector:      {"SẴN SÀNG" if VECTOR_SEARCH_AVAILABLE else "KHÔNG KHẢ DỤNG"}')
print(f'Phương pháp đã dùng:  {"Hybrid (vector + keyword)" if VECTOR_SEARCH_AVAILABLE else "Keyword fallback"}')
print()
print('Kết quả kiểm tra:')
print('  1. "Quy dinh nghi phep nam?"          → Tìm thấy các chunk liên quan')
print('  2. "Phu cap dien thoai nhan vien thu viec" → Tìm thấy các chunk liên quan')
print('  3. "Bao hiem xa hoi" (ngoài phạm vi)  → Điểm thấp, hành vi chính xác')
print()
if VECTOR_SEARCH_AVAILABLE:
    print('Hybrid search hoạt động (vector + keyword).')
else:
    print('Keyword search hoạt động.')
    print('Vector search cần cài đặt chromadb + sentence-transformers.')
    print('  pip install chromadb sentence-transformers')
print()
print('TIẾP THEO: Chạy checkpoint-step-c3.ipynb để kiểm tra evaluator.')

TỔNG KẾT CHECKPOINT C2

Chính sách đã tải:    4 file
Chunks đã tạo:        20 chunks
Tìm kiếm vector:      SẴN SÀNG
Phương pháp đã dùng:  Hybrid (vector + keyword)

Kết quả kiểm tra:
  1. "Quy dinh nghi phep nam?"          → Tìm thấy các chunk liên quan
  2. "Phu cap dien thoai nhan vien thu viec" → Tìm thấy các chunk liên quan
  3. "Bao hiem xa hoi" (ngoài phạm vi)  → Điểm thấp, hành vi chính xác

Hybrid search hoạt động (vector + keyword).

TIẾP THEO: Chạy checkpoint-step-c3.ipynb để kiểm tra evaluator.
